2) Demand by weekday vs weekend
Google’s tutorial itself separates weekday and weekend behavior, which is a strong signal for customer segmentation and planning.  
SELECT
  CASE
    WHEN EXTRACT(DAYOFWEEK FROM start_date) IN (1, 7) THEN 'Weekend'
    ELSE 'Weekday'
  END AS day_type,
  COUNT(*) AS total_rides,
  ROUND(AVG(duration) / 60, 2) AS avg_duration_mins
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY day_type;

 
Business insight: weekdays often reflect commuting demand; weekends often reflect leisure demand.

SELECT
b.day_of_week as weekday,
sum(a.hire_count)/100000 AS hire_count
FROM `bigdata-488403.london_test.fact_bike_daily_usage` a
LEFT JOIN `bigdata-488403.london_bicycles_star.dim_date` b
on a.date_value = b.date_value
GROUP BY weekday
ORDER BY hire_count DESC



 
3) Top origin stations
This shows where demand starts, useful for station capacity and replenishment.
SELECT
  start_station_name,
  COUNT(*) AS ride_starts
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY start_station_name
ORDER BY ride_starts DESC;

Business insight: identify high-pressure stations for dock expansion or faster turnaround.


SELECT
  station_name as start_station_name,
  sum(total_start_count) AS ride_starts
FROM `bigdata-488403.london_bicycles_star.dim_station` 
GROUP BY start_station_name
ORDER BY ride_starts DESC

 


4) Top destination stations
This helps spot inbound congestion and where empty docks become a problem.
SELECT
  end_station_name,
  COUNT(*) AS ride_ends
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY end_station_name
ORDER BY ride_ends DESC
;

Business insight: origin and destination rankings together show directional flow, not just popularity.

SELECT
  station_name as end_station_name,
  sum(total_end_count) AS ride_ends
FROM `bigdata-488403.london_bicycles_star.dim_station`
GROUP BY end_station_name
ORDER BY ride_ends DESC Limit 10


 

5) Most common station-to-station routes
This is useful for route planning, commuter corridor detection, and targeted promotions.
SELECT
  start_station_name,
  end_station_name,
  COUNT(*) AS trip_count,
  ROUND(AVG(duration) / 60, 2) AS avg_duration_mins
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY start_station_name, end_station_name
ORDER BY trip_count DESC
;
Business insight: reveals the highest-volume travel corridors across London.

SELECT
  --a.start_station_id,
  --a.end_station_id,
  b.station_name as start_station_name,
  c.station_name as end_station_name,
  COUNT(*) AS trip_count,
  ROUND(AVG(duration_minutes), 2) AS avg_duration_mins
FROM `bigdata-488403.london_bicycles_star.fact_hire` a
LEFT JOIN `bigdata-488403.london_bicycles_star.dim_station` b
on a.start_station_id = b.station_id 
LEFT JOIN `bigdata-488403.london_bicycles_star.dim_station` c
on a.end_station_id = c.station_id
--where a.start_station_id <> a.end_station_id
GROUP BY start_station_id, end_station_id, start_station_name, end_station_name
ORDER BY trip_count DESC LIMIT 10
 

 
6) Monthly trend and seasonality
This helps understand growth, seasonality, and campaign timing.
SELECT
  DATE_TRUNC(DATE(start_date), MONTH) AS month,
  COUNT(*) AS total_rides,
  ROUND(AVG(duration) / 60, 2) AS avg_duration_mins
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY month
ORDER BY month;
Business insight: use this to distinguish structural growth from seasonal spikes.
SELECT
b.month as month,
sum(a.hire_count)/100000 AS hire_count,
AVG(a.cumulative_duration_minutes) AS avg_cumulative_duration_mins
FROM `bigdata-488403.london_test.fact_bike_daily_usage` a
LEFT JOIN `bigdata-488403.london_bicycles_star.dim_date` b
on a.date_value = b.date_value
GROUP BY month
ORDER BY month DESC
 
7) Station net flow
This tells you which stations tend to empty out and which stations tend to fill up.
WITH starts AS (
  SELECT start_station_name AS station_name, COUNT(*) AS start_count
  FROM `bigquery-public-data.london_bicycles.cycle_hire`
  GROUP BY start_station_name
),
ends AS (
  SELECT end_station_name AS station_name, COUNT(*) AS end_count
  FROM `bigquery-public-data.london_bicycles.cycle_hire`
  GROUP BY end_station_name
)
SELECT
  COALESCE(s.station_name, e.station_name) AS station_name,
  COALESCE(start_count, 0) AS rides_started,
  COALESCE(end_count, 0) AS rides_ended,
  COALESCE(end_count, 0) - COALESCE(start_count, 0) AS net_inflow
FROM starts s
FULL OUTER JOIN ends e
  ON s.station_name = e.station_name
ORDER BY net_inflow DESC;
Business insight: stations with large negative net flow likely need more frequent bike replenishment.

WITH starts AS (
  SELECT b.station_name as start_station_name, COUNT(*) AS start_count
  FROM `bigdata-488403.london_bicycles_star.fact_hire` a
  LEFT JOIN `bigdata-488403.london_bicycles_star.dim_station` b
  on a.start_station_id = b.station_id
  GROUP BY start_station_name
),
ends AS (
  SELECT d.station_name as end_station_name, COUNT(*) AS end_count
  FROM `bigdata-488403.london_bicycles_star.fact_hire` c
  LEFT JOIN `bigdata-488403.london_bicycles_star.dim_station` d
on c.end_station_id = d.station_id
  GROUP BY end_station_name
)
SELECT
  COALESCE(s.start_station_name, e.end_station_name) AS station_name,
  COALESCE(start_count, 0) AS rides_started,
  COALESCE(end_count, 0) AS rides_ended,
  COALESCE(end_count, 0) - COALESCE(start_count, 0) AS net_inflow
FROM starts s
FULL OUTER JOIN ends e
  ON s.start_station_name = e.end_station_name
ORDER BY net_inflow DESC;
 


8) Average trip duration by station
Google’s example uses AVG(duration) and COUNT(duration) as important analytical features. 
SELECT
  start_station_name,
  COUNT(*) AS trips,
  ROUND(AVG(duration) / 60, 2) AS avg_duration_mins
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY start_station_name
HAVING trips >= 100
ORDER BY avg_duration_mins DESC;

Business insight: very long durations may indicate tourist or leisure usage rather than commuter usage.


SELECT
  b.station_name as start_station_name,
  COUNT(*) AS trips,
  ROUND(AVG(duration_minutes), 2) AS avg_duration_mins
  FROM `bigdata-488403.london_bicycles_star.fact_hire` a
  LEFT JOIN `bigdata-488403.london_bicycles_star.dim_station` b
  on a.start_station_id = b.station_id
  GROUP BY start_station_name LIMIT 100;


 


